# MeAJOR Dataset Preparation

This notebook prepares the MeAJOR dataset for text-only phishing email classification. It covers data inspection, text field construction, duplicate handling, and the final 60/40 train-test split.

In [12]:
import pandas as pd
from pathlib import Path

## 1. Load and inspect the raw dataset

This section loads the MeAJOR dataset and performs an initial inspection of its size, schema, and example rows.

In [13]:
data_path = Path("/Users/ziliang/Downloads/cm3203-phishing-nlp/data/raw/meajor/meajor_cleaned_preprocessed.parquet.gzip")
df = pd.read_parquet(data_path)
df.shape

(108685, 20)

In [14]:
df.columns.tolist()

['sender',
 'sender_domain',
 'receiver',
 'receiver_domain',
 'date',
 'subject',
 'content_types',
 'body',
 'urls',
 'url_count',
 'url_length_max',
 'url_length_avg',
 'url_subdom_max',
 'url_subdom_avg',
 'attachment_count',
 'has_attachments',
 'attachment_types',
 'language',
 'source',
 'label']

In [15]:
df.head(5)

,sender,sender_domain,receiver,receiver_domain,date,subject,content_types,body,urls,url_count,url_length_max,url_length_avg,url_subdom_max,url_subdom_avg,attachment_count,has_attachments,attachment_types,language,source,label
0,d66e9e64b006d6bca649f1c945129c42c43836872b2ead...,enron.com,35c5a9fb9fba3b8737ed7cef2a87e427a73db4fca85f6b...,enron.com,2001-06-29 09:37:04-05:00,[ORGANIZATION] failover plan.,text/plain,"Hi [NAME], \n\nTonight we are rolling out a n...",None,0.0,0.0,0.0,0.0,0.0,0.0,False,None,en,trec5,0.0
1,0907d5c64598aa2639154ed4e1556be615669e40052a1f...,enron.com,aa2c35499eae5999bf6080453cc719a891da2bb0c3803d...,enron.com,2001-06-29 08:39:30-05:00,RE: Intranet Site,text/plain,"[NAME] r these new?\tIntranet Site\n\n[NAME],\...",http://eastpower.dev.corp.enron.com/summary/pj...,3.0,60.0,58.0,3.0,3.0,0.0,False,None,en,trec5,0.0
2,7c3201a5ff8c5985218f1e3f11e330dc0242bbd28c6c20...,enron.com,a736837579feb601fbf6c0657d3d93689774afa6491bb9...,enron.com;enron.com,2001-06-29 10:35:17-05:00,FW: [ORGANIZATION] Company information,text/plain,"[NAME]/[NAME],\n\nWe are currently trading und...",None,0.0,0.0,0.0,0.0,0.0,0.0,False,None,en,trec5,0.0
3,8531d54a169c4af106b9ea2165d4986b8cc10fc0a6bb9b...,enron.com,765a3ec4a67e40118d22de5729b05d090a1b59cb443bf6...,enron.com;enron.com,2001-06-29 10:40:02-05:00,New Master Physical,text/plain,[NAME] and [NAME] -\n\nAttached is a worksheet...,None,0.0,0.0,0.0,0.0,0.0,0.0,False,None,en,trec5,0.0
4,7c3201a5ff8c5985218f1e3f11e330dc0242bbd28c6c20...,enron.com,ce418c97ac415706338972e1dbbd99ebb8c617b5c937a3...,enron.com;enron.com;enron.com,2001-06-29 10:48:00-05:00,FW: [ORGANIZATION]/Mirant GISB,text/plain,FYI. Below is a copy of my communication with ...,None,0.0,0.0,0.0,0.0,0.0,0.0,False,None,en,trec5,0.0


## 2. Examine data quality and structure

This section checks missing values, column variability, and random sample rows to better understand the dataset before preprocessing decisions are made.

In [16]:
df.isna().sum().sort_values(ascending=False)

attachment_types    107019
urls                 44633
receiver_domain       2371
subject               1455
sender_domain          494
label                    1
source                   1
language                 1
content_types            1
body                     1
url_subdom_avg           0
has_attachments          0
attachment_count         0
sender                   0
url_subdom_max           0
url_length_avg           0
url_count                0
date                     0
receiver                 0
url_length_max           0
dtype: int64

In [17]:
for col in df.columns:
    unique_count = df[col].nunique(dropna=False)
    print(col, unique_count)

sender 55563
sender_domain 28465
receiver 17581
receiver_domain 6367
date 107474
subject 68487
content_types 17
body 103519
urls 38353
url_count 153
url_length_max 255
url_length_avg 4497
url_subdom_max 6
url_subdom_avg 573
attachment_count 13
has_attachments 2
attachment_types 146
language 244
source 4
label 3


In [18]:
df.sample(5, random_state=42)

,sender,sender_domain,receiver,receiver_domain,date,subject,content_types,body,urls,url_count,url_length_max,url_length_avg,url_subdom_max,url_subdom_avg,attachment_count,has_attachments,attachment_types,language,source,label
93166,781eb0d8ec79f0655e11a45ba63d7cceaad76283491c12...,cajau.com,c8151cb6e18758f810572bfc19bd74b6376f78c4cb4d29...,flax9.uwaterloo.ca,2007-06-06 16:11:02-09:00,Must have pills,text/plain,[ORGANIZATION] is dedicated to providing you w...,http://hugeraise.hk/\nhttp://hugeraise.hk/,2.0,20.0,20.0,0.0,0.0,0.0,False,None,en,trec7,1.0
17303,35c9fd53a8f4a6f2a0023d6ec222f8c9050e5dfb525539...,wt.net,484d7e37f87d4bcfbbac726025d65df7cd405efb32f3b5...,yahoogroups.com,2001-10-25 11:19:14-06:00,Re: *EMCA* Mosquitoes,text/plain,All the many neighbors in the energy industry ...,http://us.click.yahoo.com/Gi0tnD/bQ8CAA/ySSFAA...,2.0,56.0,44.5,2.0,1.5,0.0,False,None,en,trec5,0.0
40628,25c90517b971ddb8f09623dfc512d125baaa45724a2488...,dfwfinancials.com,dbbeb4550fea80360a092309fa11105daedbf168e8862f...,enron.com,2002-02-19 14:26:11-06:00,Where do we send the [FINANCIAL_INFO]?,text/plain,"Dear Homeowner,\n\nYou have been pre-approved ...",http://www.jhaevb.info\nhttp://nyr.jhaevb.info,2.0,22.0,22.0,1.0,1.0,0.0,False,None,en,trec5,1.0
64788,355c34c8329fa5da5b603477d5f25b7480387280b2284f...,samba.org,f2d78073ba3c44ba2e628c37551de8435b1f699fc3ace5...,lists.samba.org,2007-04-08 21:32:26-04:00,Re: svn commit: samba r[REFERENCE_NUMBER] - in...,text/plain,<|EMAIL|> writes:\n\nAuthor: [NAME]\nDate: [DA...,http://websvn.samba.org/cgi-bin/viewcvs.cgi?vi...,1.0,73.0,73.0,1.0,1.0,0.0,False,None,en,trec7,0.0
38042,a3c6fa1369692d6efdc2147cb5f426f785dad71a31300c...,comcast.net,ce44efa99ef454f4ea68835b3eb75d4c724f43e9ac9eff...,enron.com,2002-01-31 06:38:19-05:00,72% off for All New Software. entrepreneurial...,text/html,"Through, took house brought paint. Book busy u...",http://www.shutoff.linchidmch.com/?dYfif.JtJhk...,1.0,59.0,59.0,2.0,2.0,0.0,False,None,en,trec5,1.0


## 3. Check labels and duplicates

This section verifies that the task is binary, identifies missing labels, and checks for duplicate rows and duplicate final texts.

In [19]:
print("LABEL VALUE COUNTS")
print(df["label"].value_counts(dropna=False))
print()

print("LABEL UNIQUE VALUES")
print(df["label"].unique())
print()

print("MISSING VALUES: subject, body, label")
print(df[["subject", "body", "label"]].isna().sum())
print()

print("EXACT DUPLICATE ROWS")
print(df.duplicated().sum())
print()

final_text_duplicates = (
    df.assign(
        final_text=df["subject"].fillna("").astype(str).str.strip() + " " +
                   df["body"].fillna("").astype(str).str.strip()
    )
    .assign(final_text=lambda x: x["final_text"].str.replace(r"\s+", " ", regex=True).str.strip())
    ["final_text"]
    .duplicated()
    .sum()
)

print("FINAL TEXT DUPLICATES")
print(final_text_duplicates)

LABEL VALUE COUNTS
label
0.0    60650
1.0    48034
NaN        1
Name: count, dtype: int64

LABEL UNIQUE VALUES
[ 0.  1. nan]

MISSING VALUES: subject, body, label
subject    1455
body          1
label         1
dtype: int64

EXACT DUPLICATE ROWS
13

FINAL TEXT DUPLICATES
4112


## 4. Build the modelling dataset

This section keeps only the text fields and label, constructs the final input text from subject and body, removes invalid rows, and removes duplicate text-label pairs.

In [20]:
import pandas as pd

meajor = df[["subject", "body", "label"]].copy()

meajor = meajor.dropna(subset=["label"])

meajor["subject"] = meajor["subject"].fillna("").astype(str)
meajor["body"] = meajor["body"].fillna("").astype(str)

meajor["text"] = (
    "Subject: " + meajor["subject"].str.strip() +
    " Body: " + meajor["body"].str.strip()
)

meajor["text"] = meajor["text"].str.replace(r"\s+", " ", regex=True).str.strip()
meajor["label"] = meajor["label"].astype(int)

meajor = meajor[meajor["text"].str.len() > 0].copy()

before_dedup = len(meajor)
meajor = meajor.drop_duplicates(subset=["text", "label"]).copy()
after_dedup = len(meajor)

print("Rows before dedup:", before_dedup)
print("Rows after dedup:", after_dedup)
print("Rows removed:", before_dedup - after_dedup)
print()
print(meajor["label"].value_counts())
print()
print(meajor.head())

Rows before dedup: 108684
Rows after dedup: 104572
Rows removed: 4112

label
0    57947
1    46625
Name: count, dtype: int64

                                  subject  \
0           [ORGANIZATION] failover plan.   
1                       RE: Intranet Site   
2  FW: [ORGANIZATION] Company information   
3                     New Master Physical   
4          FW: [ORGANIZATION]/Mirant GISB   

                                                body  label  \
0  Hi [NAME],  \n\nTonight we are rolling out a n...      0   
1  [NAME] r these new?\tIntranet Site\n\n[NAME],\...      0   
2  [NAME]/[NAME],\n\nWe are currently trading und...      0   
3  [NAME] and [NAME] -\n\nAttached is a worksheet...      0   
4  FYI. Below is a copy of my communication with ...      0   

                                                text  
0  Subject: [ORGANIZATION] failover plan. Body: H...  
1  Subject: RE: Intranet Site Body: [NAME] r thes...  
2  Subject: FW: [ORGANIZATION] Company informatio...  
3  S

## 5. Create and save the final split

This section creates the stratified 60/40 train-test split and saves the processed dataset files for later modelling.

In [21]:
from sklearn.model_selection import train_test_split
from pathlib import Path

RANDOM_SEED = 42

train_df, test_df = train_test_split(
    meajor[["text", "label"]],
    test_size=0.40,
    random_state=RANDOM_SEED,
    stratify=meajor["label"]
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print()

print("Train label distribution:")
print(train_df["label"].value_counts())
print()
print(train_df["label"].value_counts(normalize=True))
print()

print("Test label distribution:")
print(test_df["label"].value_counts())
print()
print(test_df["label"].value_counts(normalize=True))

Train shape: (62743, 2)
Test shape: (41829, 2)

Train label distribution:
label
0    34768
1    27975
Name: count, dtype: int64

label
0    0.554134
1    0.445866
Name: proportion, dtype: float64

Test label distribution:
label
0    23179
1    18650
Name: count, dtype: int64

label
0    0.554137
1    0.445863
Name: proportion, dtype: float64


## 6. Outcome

The MeAJOR dataset has now been reduced to a cleaned text-only format and saved as reproducible train and test splits ready for baseline modelling.

In [22]:
output_dir = Path("../data/processed/meajor")
output_dir.mkdir(parents=True, exist_ok=True)

meajor[["text", "label"]].to_parquet(output_dir / "meajor_text_label_dedup.parquet", index=False)
train_df.to_parquet(output_dir / "meajor_train_60.parquet", index=False)
test_df.to_parquet(output_dir / "meajor_test_40.parquet", index=False)

print("Saved:")
print(output_dir / "meajor_text_label_dedup.parquet")
print(output_dir / "meajor_train_60.parquet")
print(output_dir / "meajor_test_40.parquet")

Saved:
../data/processed/meajor/meajor_text_label_dedup.parquet
../data/processed/meajor/meajor_train_60.parquet
../data/processed/meajor/meajor_test_40.parquet
